In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "chibifire/kaggle-womens-ecom-clothing-reviews"
)

print(dataset)

README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.49MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/23486 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['row_id', 'clothing_id', 'age', 'rating', 'recommended_ind', 'positive_feedback_count', 'title', 'review_text', 'division_name', 'department_name', 'class_name'],
        num_rows: 23486
    })
})


In [ ]:
import pandas as pd

df = dataset["train"].to_pandas()

In [ ]:
df.shape

(23486, 11)

In [ ]:
df.columns

Index(['row_id', 'clothing_id', 'age', 'rating', 'recommended_ind',
       'positive_feedback_count', 'title', 'review_text', 'division_name',
       'department_name', 'class_name'],
      dtype='object')

In [ ]:
df[["rating", "title", "review_text"]].head()

,rating,title,review_text
0,4,None,Absolutely wonderful - silky and sexy and comf...
1,5,None,Love this dress! it's sooo pretty. i happene...
2,3,Some major design flaws,I had such high hopes for this dress and reall...
3,5,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl..."
4,5,Flattering shirt,This shirt is very flattering to all due to th...


In [ ]:
df["rating"].value_counts().sort_index()

,count
rating,
1,842
2,1565
3,2871
4,5077
5,13131


In [ ]:
df[["rating", "title", "review_text"]].isnull().sum()

,0
rating,0
title,3810
review_text,845


In [ ]:
df = df[["review_text", "rating"]].copy()

In [ ]:
df = df.dropna(subset=["review_text"])
df = df[df["review_text"].str.strip() != ""]

In [ ]:
def rating_to_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["rating"].apply(rating_to_sentiment)

In [ ]:
df.shape

(22641, 3)

In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
positive,17448
neutral,2823
negative,2370


In [ ]:
(df["sentiment"].value_counts(normalize=True) * 100).round(2)

,proportion
sentiment,
positive,77.06
neutral,12.47
negative,10.47


In [ ]:
df.head()

,review_text,rating,sentiment
0,Absolutely wonderful - silky and sexy and comf...,4,positive
1,Love this dress! it's sooo pretty. i happene...,5,positive
2,I had such high hopes for this dress and reall...,3,neutral
3,"I love, love, love this jumpsuit. it's fun, fl...",5,positive
4,This shirt is very flattering to all due to th...,5,positive


In [ ]:
from sklearn.model_selection import train_test_split

X = df["review_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train nümunələri:", len(X_train))
print("Test nümunələri:", len(X_test))

print("\nTrain sinif faizləri:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest sinif faizləri:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train nümunələri: 18112
Test nümunələri: 4529

Train sinif faizləri:
sentiment
positive    77.06
neutral     12.47
negative    10.47
Name: proportion, dtype: float64

Test sinif faizləri:
sentiment
positive    77.06
neutral     12.48
negative    10.47
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
import os

project_dir = "data"
os.makedirs(project_dir, exist_ok=True)

train_df = pd.DataFrame({
    "review_text": X_train,
    "sentiment": y_train
})

test_df = pd.DataFrame({
    "review_text": X_test,
    "sentiment": y_test
})

train_df.to_csv(
    f"{project_dir}/train.csv",
    index=False
)

test_df.to_csv(
    f"{project_dir}/test.csv",
    index=False
)

print("Fayllar uğurla saxlanıldı!")
print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Qovluq:", project_dir)

Fayllar uğurla saxlanıldı!
Train: (18112, 2)
Test: (4529, 2)
Qovluq: /content/drive/MyDrive/Product_Review_Sentiment_Classifier


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix
)

In [ ]:
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight=None,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, min_df=2,
                                 ngram_range=(1, 2))),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
# Test rəyləri üçün proqnozlar
y_pred = baseline_model.predict(X_test)

# Accuracy
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

# Hər sinif üçün Precision, Recall və F1-score
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred,
    labels=["negative", "neutral", "positive"]
))

Accuracy: 0.8251

Classification Report:
              precision    recall  f1-score   support

    negative     0.6403    0.4093    0.4994       474
     neutral     0.4463    0.2354    0.3082       565
    positive     0.8681    0.9771    0.9194      3490

    accuracy                         0.8251      4529
   macro avg     0.6516    0.5406    0.5757      4529
weighted avg     0.7917    0.8251    0.7992      4529


Confusion Matrix:
[[ 194  109  171]
 [  85  133  347]
 [  24   56 3410]]


In [ ]:
balanced_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

balanced_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, min_df=2,
                                 ngram_range=(1, 2))),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_balanced = balanced_model.predict(X_test)

print("Balanced Accuracy:", round(
    accuracy_score(y_test, y_pred_balanced), 4
))

print(classification_report(
    y_test,
    y_pred_balanced,
    digits=4
))

Balanced Accuracy: 0.7995
              precision    recall  f1-score   support

    negative     0.4930    0.5949    0.5392       474
     neutral     0.3727    0.5027    0.4280       565
    positive     0.9562    0.8754    0.9140      3490

    accuracy                         0.7995      4529
   macro avg     0.6073    0.6576    0.6271      4529
weighted avg     0.8349    0.7995    0.8141      4529



In [ ]:
!pip install -q transformers accelerate

import torch
import transformers

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Transformers version: 5.20.2
PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

project_dir = "/content/drive/MyDrive/Product_Review_Sentiment_Classifier"


train_df = pd.read_csv(f"{project_dir}/train.csv")
test_df = pd.read_csv(f"{project_dir}/test.csv")


train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["sentiment"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain sinifləri:")
print(train_df["sentiment"].value_counts())

print("\nValidation sinifləri:")
print(val_df["sentiment"].value_counts())

Train: (16300, 2)
Validation: (1812, 2)
Test: (4529, 2)

Train sinifləri:
sentiment
positive    12562
neutral      2032
negative     1706
Name: count, dtype: int64

Validation sinifləri:
sentiment
positive    1396
neutral      226
negative     190
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

example = "The dress is beautiful"

tokens = tokenizer.tokenize(example)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original text:", example)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original text: The dress is beautiful
Tokens: ['the', 'dress', 'is', 'beautiful']
Token IDs: [1996, 4377, 2003, 3376]


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

def prepare_dataset(dataframe):
    data = dataframe[["review_text", "sentiment"]].copy()

    # Sentiment sözlərini ədədi etiketlərə çeviririk
    data["labels"] = data["sentiment"].map(label2id)

    # Hugging Face Dataset formatına çeviririk
    dataset = Dataset.from_pandas(
        data[["review_text", "labels"]],
        preserve_index=False
    )

    # Mətnləri tokenlərə çeviririk
    def tokenize_function(batch):
        return tokenizer(
            batch["review_text"],
            truncation=True,
            max_length=256
        )

    dataset = dataset.map(
        tokenize_function,
        batched=True
    )

    # Modelə lazım olan sütunları saxlayırıq
    dataset = dataset.remove_columns(["review_text"])

    return dataset

train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)
test_dataset = prepare_dataset(test_df)

print("Train nümunəsi:")
print(train_dataset[0])

print("\nDataset ölçüləri:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Map:   0%|          | 0/16300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1812 [00:00<?, ? examples/s]

Map:   0%|          | 0/4529 [00:00<?, ? examples/s]

Train nümunəsi:
{'labels': 2, 'input_ids': [101, 3737, 2003, 6581, 1012, 1045, 5993, 2007, 2178, 12027, 2008, 1996, 2327, 3216, 2312, 1012, 1045, 2572, 2788, 1037, 5396, 1010, 2021, 1045, 2001, 5742, 1999, 2009, 1012, 1045, 2444, 1999, 7397, 1998, 2052, 2025, 2113, 2129, 2000, 2709, 2009, 2061, 1045, 2572, 2074, 4363, 2009, 1998, 4147, 2009, 2006, 2216, 4658, 13971, 2420, 1012, 1045, 1005, 1049, 2469, 2009, 2052, 2145, 2298, 2465, 2100, 2138, 1997, 1996, 2640, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Dataset ölçüləri:
Train: 16300
Validation: 1812
Te

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

# Hər batch daxilində mətnləri ən uzun nümunəyə uyğun padding edirik
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Model uğurla yükləndi!")
print("Sinif sayı:", model.config.num_labels)
print("Etiketlər:", model.config.id2label)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model uğurla yükləndi!
Sinif sayı: 3
Etiketlər: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments, Trainer

# Modelin nəticələrini hesablayan funksiya
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Modelin hər sinif üçün verdiyi ballardan ən yüksək olanı seçirik
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

# Təlim parametrləri
training_args = TrainingArguments(
    output_dir=f"{project_dir}/distilbert_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    report_to="none",
    seed=42
)

print("Təlim parametrləri hazırdır.")
print("Epoch:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)

Təlim parametrləri hazırdır.
Epoch: 3
Train batch size: 16
Learning rate: 2e-05


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer hazırdır.")
print("Train nümunələrinin sayı:", len(train_dataset))
print("Validation nümunələrinin sayı:", len(val_dataset))

Trainer hazırdır.
Train nümunələrinin sayı: 16300
Validation nümunələrinin sayı: 1812


In [ ]:
training_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.338153,0.383923,0.841060,0.662600,0.675608,0.668333
2,0.302543,0.409266,0.843819,0.671926,0.657142,0.662929
3,0.246288,0.449465,0.837748,0.673586,0.662685,0.666648


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print("Ən yaxşı checkpoint:", trainer.state.best_model_checkpoint)
print("Ən yaxşı validation metric:", trainer.state.best_metric)
print("Model hazırda GPU-dadır:", next(model.parameters()).is_cuda)

Ən yaxşı checkpoint: /content/drive/MyDrive/Product_Review_Sentiment_Classifier/distilbert_checkpoints/checkpoint-1019
Ən yaxşı validation metric: 0.6683331097913986
Model hazırda GPU-dadır: True


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
import numpy as np

# Test datasında proqnozlar
test_output = trainer.predict(test_dataset)

# Modelin proqnozlaşdırdığı sinif ID-ləri
y_pred_transformer = np.argmax(
    test_output.predictions,
    axis=-1
)

# Həqiqi test etiketləri
y_test_transformer = test_output.label_ids

# Rəqəmləri sentiment adlarına çeviririk
id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

y_pred_labels = [id2label[i] for i in y_pred_transformer]
y_true_labels = [id2label[i] for i in y_test_transformer]

print("Test Accuracy:", accuracy_score(
    y_true_labels, y_pred_labels
))

print("\nClassification Report:")
print(classification_report(
    y_true_labels,
    y_pred_labels,
    labels=["negative", "neutral", "positive"],
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_true_labels,
    y_pred_labels,
    labels=["negative", "neutral", "positive"]
))

Test Accuracy: 0.8478692868182822

Classification Report:
              precision    recall  f1-score   support

    negative     0.5908    0.6519    0.6199       474
     neutral     0.4590    0.4265    0.4422       565
    positive     0.9451    0.9427    0.9439      3490

    accuracy                         0.8479      4529
   macro avg     0.6650    0.6737    0.6687      4529
weighted avg     0.8474    0.8479    0.8474      4529

Confusion Matrix:
[[ 309  127   38]
 [ 171  241  153]
 [  43  157 3290]]


In [ ]:
# Test rəylərini və Transformer proqnozları
error_df = test_df[["review_text", "sentiment"]].copy()
error_df["predicted_sentiment"] = y_pred_labels

# Yalnız səhv proqnozlar
errors = error_df[
    error_df["sentiment"] != error_df["predicted_sentiment"]
].copy()

print("Səhv proqnozların sayı:", len(errors))

# İlk 5 səhv nümunə
for i, row in errors.head(5).iterrows():
    print("\n" + "=" * 80)
    print("Həqiqi sentiment:", row["sentiment"])
    print("Modelin proqnozu:", row["predicted_sentiment"])
    print("Rəy:", row["review_text"])

Səhv proqnozların sayı: 689

Həqiqi sentiment: neutral
Modelin proqnozu: positive
Rəy: Stretch out fast

5,8 130lb i order size 26 which is my usually size. they fit prefect for about 3hr and then stretched out feeling not right on my body. maybe i should have gone down a  sizes. even so i feel like the $$$ and quality don't add up but they are very soft and comfortable, i wouldn't buy them again.

Həqiqi sentiment: neutral
Modelin proqnozu: positive
Rəy: This is one of the best potential dresses i've seen from retailer, but it needs a do-over. i'm usually a 12 in this style of dress, but i sized up to a 16 (the largest size) and couldn't even button the top. even if i had been able to do so, i would have worn it as a tunic. i am 5'5'' and the largest size would have been a racy mini skirt at best on me if worn as a dress. this dress is fit for only the tiniest and most small-breasted of people. that said, if the company were to try again ne

Həqiqi sentiment: positive
Modelin proqnozu

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["sentiment"].map(label2id).values
)

print("Siniflər:", classes)
print("Sinif çəkiləri:", class_weights)

Siniflər: [0 1 2]
Sinif çəkiləri: [3.18483783 2.67388451 0.43252136]


In [ ]:

import torch
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(
            class_weights, dtype=torch.float
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        weights = self.class_weights.to(outputs.logits.device)

        loss_function = nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_function(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss


In [ ]:
from transformers import AutoModelForSequenceClassification

# Modeli yenidən ilkin pretrained checkpoint-dən yükləyirik
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

# Sinif çəkilərini istifadə edən Trainer
weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
weighted_training_result = weighted_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.614680,0.699124,0.791943,0.590214,0.673823,0.617280
2,0.530982,0.682147,0.826711,0.655574,0.716417,0.680215
3,0.417193,0.777243,0.824503,0.650840,0.704014,0.671938


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

val_output = weighted_trainer.predict(val_dataset)

val_predictions = np.argmax(
    val_output.predictions, axis=1
)

val_true_labels = val_output.label_ids

print(classification_report(
    val_true_labels,
    val_predictions,
    labels=[0, 1, 2],
    target_names=["negative", "neutral", "positive"],
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

    negative     0.5948    0.7263    0.6540       190
     neutral     0.4047    0.5354    0.4610       226
    positive     0.9672    0.8875    0.9257      1396

    accuracy                         0.8267      1812
   macro avg     0.6556    0.7164    0.6802      1812
weighted avg     0.8580    0.8267    0.8392      1812



In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Əvvəlki, class weighting istifadə etməyən model
original_val_output = trainer.predict(val_dataset)

original_val_predictions = np.argmax(
    original_val_output.predictions,
    axis=1
)

original_val_labels = original_val_output.label_ids

print("ORIGINAL DISTILBERT — VALIDATION")

print(classification_report(
    original_val_labels,
    original_val_predictions,
    labels=[0, 1, 2],
    target_names=["negative", "neutral", "positive"],
    digits=4,
    zero_division=0
))

ORIGINAL DISTILBERT — VALIDATION
              precision    recall  f1-score   support

    negative     0.6085    0.6789    0.6418       190
     neutral     0.4413    0.4159    0.4282       226
    positive     0.9380    0.9319    0.9350      1396

    accuracy                         0.8411      1812
   macro avg     0.6626    0.6756    0.6683      1812
weighted avg     0.8415    0.8411    0.8410      1812



In [ ]:
from sklearn.metrics import classification_report
import numpy as np

weighted_test_output = weighted_trainer.predict(test_dataset)

weighted_test_predictions = np.argmax(
    weighted_test_output.predictions,
    axis=1
)

print("WEIGHTED DISTILBERT — TEST")

print(classification_report(
    weighted_test_output.label_ids,
    weighted_test_predictions,
    labels=[0, 1, 2],
    target_names=["negative", "neutral", "positive"],
    digits=4,
    zero_division=0
))

WEIGHTED DISTILBERT — TEST
              precision    recall  f1-score   support

    negative     0.6098    0.7089    0.6556       474
     neutral     0.4127    0.5522    0.4724       565
    positive     0.9699    0.8954    0.9312      3490

    accuracy                         0.8331      4529
   macro avg     0.6641    0.7188    0.6864      4529
weighted avg     0.8627    0.8331    0.8451      4529



In [ ]:

import os
import time
import torch

# 1. Modelin parametr sayı
parameter_count = sum(
    p.numel() for p in weighted_model.parameters()
)

# 2. Modelin yaddaşda tutduğu parametr ölçüsü
parameter_size_mb = sum(
    p.numel() * p.element_size()
    for p in weighted_model.parameters()
) / (1024 ** 2)

print(f"Parameter sayı: {parameter_count:,}")
print(f"Parametrlərin yaddaş ölçüsü: {parameter_size_mb:.2f} MB")

# 3. Bir rəy üçün proqnoz funksiyası
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    device = next(weighted_model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    weighted_model.eval()

    with torch.no_grad():
        outputs = weighted_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)
        prediction_id = outputs.logits.argmax(dim=1).item()

    return {
        "sentiment": id2label[prediction_id],
        "confidence": probabilities[0, prediction_id].item()
    }

# 4. Nümunə rəy
sample_review = (
    "The material is soft and comfortable, "
    "but the size is too small."
)

print(predict_sentiment(sample_review))


Parameter sayı: 66,955,779
Parametrlərin yaddaş ölçüsü: 255.42 MB
{'sentiment': 'neutral', 'confidence': 0.7489339709281921}


In [ ]:

import time
import torch

reviews = test_df["review_text"].head(100).tolist()

# GPU əməliyyatlarının tamamlanmasını gözləyirik
if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.perf_counter()

for review in reviews:
    predict_sentiment(review)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed_time = time.perf_counter() - start_time

print(f"Rəylərin sayı: {len(reviews)}")
print(f"Ümumi vaxt: {elapsed_time:.3f} saniyə")
print(f"Bir rəy üçün orta vaxt: {elapsed_time / len(reviews):.4f} saniyə")
print(f"Saniyədə rəy sayı: {len(reviews) / elapsed_time:.2f}")


Rəylərin sayı: 100
Ümumi vaxt: 1.400 saniyə
Bir rəy üçün orta vaxt: 0.0140 saniyə
Saniyədə rəy sayı: 71.44
